# Sign Language Numbers Recognition - Training Model

This notebook trains a model to recognize sign language numbers from video data.

## Steps:
1. Setup environment and install dependencies
2. Upload/load video data
3. Extract features using MediaPipe
4. Prepare dataset
5. Train the model
6. Evaluate and save the model


## Step 1: Setup and Install Dependencies


In [ ]:
# Install required packages
!pip install -q mediapipe opencv-python tensorflow numpy scikit-learn matplotlib seaborn tqdm


In [ ]:
# Import libraries
import os
import cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"MediaPipe version: {mp.__version__}")


## Step 2: Configuration


In [ ]:
# Configuration
DATA_DIR = '/content/data_raw'  # Update this path to your data directory
OUTPUT_DIR = '/content/output'
MODEL_DIR = '/content/models'

# Create directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Model parameters
SEQUENCE_LENGTH = 30  # Number of frames to use for each video
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = None  # Will be determined from data
EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 0.001

# MediaPipe settings
mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils


## Step 3: Upload Video Data

### Option A: Upload from Google Drive
If your videos are in Google Drive, mount it and copy files:


In [ ]:
# Mount Google Drive (uncomment if using Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r '/content/drive/MyDrive/your_video_folder/*' {DATA_DIR}/


### Option B: Upload files directly
Use the file uploader below to upload your video files:


In [ ]:
# Upload files (uncomment to use)
# from google.colab import files
# uploaded = files.upload()
# # Then organize them into folders by number


## Step 4: Extract Features from Videos

We'll use MediaPipe to extract hand and pose landmarks from each video frame.


In [ ]:
def extract_landmarks_from_video(video_path, max_frames=SEQUENCE_LENGTH):
    """
    Extract hand and pose landmarks from a video file.
    
    Args:
        video_path: Path to video file
        max_frames: Maximum number of frames to extract
    
    Returns:
        landmarks: Array of landmarks (max_frames, num_landmarks, 3)
    """
    cap = cv2.VideoCapture(video_path)
    
    # Initialize MediaPipe
    hands = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )
    
    pose = mp_pose.Pose(
        static_image_mode=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )
    
    landmarks_list = []
    frame_count = 0
    
    while cap.isOpened() and frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Convert BGR to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Process with MediaPipe
        hand_results = hands.process(frame_rgb)
        pose_results = pose.process(frame_rgb)
        
        # Extract landmarks
        frame_landmarks = []
        
        # Hand landmarks (21 points per hand, 2 hands max = 42 points)
        if hand_results.multi_hand_landmarks:
            for hand_landmarks in hand_results.multi_hand_landmarks:
                for landmark in hand_landmarks.landmark:
                    frame_landmarks.extend([landmark.x, landmark.y, landmark.z])
            # Pad if only one hand detected
            if len(hand_results.multi_hand_landmarks) == 1:
                frame_landmarks.extend([0.0] * 63)  # 21 points * 3 coords
        else:
            # No hands detected, pad with zeros
            frame_landmarks.extend([0.0] * 126)  # 2 hands * 21 points * 3 coords
        
        # Pose landmarks (33 points)
        if pose_results.pose_landmarks:
            for landmark in pose_results.pose_landmarks.landmark:
                frame_landmarks.extend([landmark.x, landmark.y, landmark.z, landmark.visibility])
        else:
            # No pose detected, pad with zeros
            frame_landmarks.extend([0.0] * 132)  # 33 points * 4 coords
        
        landmarks_list.append(frame_landmarks)
        frame_count += 1
    
    cap.release()
    hands.close()
    pose.close()
    
    # Pad or truncate to max_frames
    if len(landmarks_list) < max_frames:
        # Pad with last frame
        last_frame = landmarks_list[-1] if landmarks_list else [0.0] * 258
        while len(landmarks_list) < max_frames:
            landmarks_list.append(last_frame)
    elif len(landmarks_list) > max_frames:
        # Truncate
        landmarks_list = landmarks_list[:max_frames]
    
    return np.array(landmarks_list, dtype=np.float32)


In [ ]:
def process_all_videos(data_dir, output_file='landmarks_data.pkl'):
    """
    Process all videos in the data directory and extract landmarks.
    
    Expected directory structure:
    data_dir/
        number_1/
            video1.mp4
            video2.mp4
            ...
        number_2/
            ...
    """
    X = []  # Features
    y = []  # Labels
    
    # Supported video formats
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.flv', '.wmv']
    
    # Get all number folders
    data_path = Path(data_dir)
    number_folders = sorted([f for f in data_path.iterdir() if f.is_dir()])
    
    print(f"Found {len(number_folders)} number folders")
    
    for number_folder in tqdm(number_folders, desc="Processing folders"):
        # Extract number from folder name (e.g., '1', '2', '200', '210')
        folder_name = number_folder.name
        # Try to extract number from folder name
        try:
            # Handle names like 'number_1', '1', 'num_200', etc.
            number = int(''.join(filter(str.isdigit, folder_name)))
        except:
            print(f"Warning: Could not extract number from folder {folder_name}, skipping...")
            continue
        
        # Get all video files in this folder
        video_files = [f for f in number_folder.iterdir() 
                      if f.suffix.lower() in video_extensions]
        
        print(f"\nProcessing {len(video_files)} videos for number {number}")
        
        for video_file in tqdm(video_files, desc=f"Number {number}", leave=False):
            try:
                landmarks = extract_landmarks_from_video(str(video_file))
                X.append(landmarks)
                y.append(number)
            except Exception as e:
                print(f"\nError processing {video_file}: {str(e)}")
                continue
    
    X = np.array(X)
    y = np.array(y)
    
    print(f"\nTotal samples: {len(X)}")
    print(f"Feature shape: {X.shape}")
    print(f"Unique labels: {len(np.unique(y))}")
    print(f"Labels: {sorted(np.unique(y))}")
    
    # Save processed data
    with open(os.path.join(OUTPUT_DIR, output_file), 'wb') as f:
        pickle.dump({'X': X, 'y': y}, f)
    
    print(f"\nSaved processed data to {os.path.join(OUTPUT_DIR, output_file)}")
    
    return X, y


In [ ]:
# Process all videos (this may take a while)
# Uncomment to run:
# X, y = process_all_videos(DATA_DIR)

# Or load previously processed data:
# with open(os.path.join(OUTPUT_DIR, 'landmarks_data.pkl'), 'rb') as f:
#     data = pickle.load(f)
#     X, y = data['X'], data['y']


## Step 5: Prepare Dataset for Training


In [ ]:
def prepare_dataset(X, y, test_size=0.2, val_size=0.1, random_state=42):
    """
    Prepare dataset for training with train/validation/test splits.
    """
    # Encode labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    
    # First split: train+val and test
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y_encoded, test_size=test_size, random_state=random_state, stratify=y_encoded
    )
    
    # Second split: train and val
    val_size_adjusted = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_size_adjusted, random_state=random_state, stratify=y_temp
    )
    
    # Convert to categorical for multi-class classification
    num_classes = len(np.unique(y_encoded))
    y_train_cat = keras.utils.to_categorical(y_train, num_classes)
    y_val_cat = keras.utils.to_categorical(y_val, num_classes)
    y_test_cat = keras.utils.to_categorical(y_test, num_classes)
    
    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    print(f"Test samples: {len(X_test)}")
    print(f"Number of classes: {num_classes}")
    print(f"Feature shape: {X_train.shape}")
    
    return (X_train, y_train_cat), (X_val, y_val_cat), (X_test, y_test_cat), label_encoder, num_classes


In [ ]:
# Prepare dataset (uncomment after processing videos)
# (X_train, y_train), (X_val, y_val), (X_test, y_test), label_encoder, NUM_CLASSES = prepare_dataset(X, y)

# Save label encoder
# with open(os.path.join(MODEL_DIR, 'label_encoder.pkl'), 'wb') as f:
#     pickle.dump(label_encoder, f)


## Step 6: Build and Train the Model

We'll use an LSTM-based model to handle the temporal sequence of landmarks.


In [ ]:
def create_lstm_model(input_shape, num_classes, learning_rate=LEARNING_RATE):
    """
    Create an LSTM-based model for sequence classification.
    
    Args:
        input_shape: Shape of input (sequence_length, num_features)
        num_classes: Number of output classes
        learning_rate: Learning rate for optimizer
    
    Returns:
        model: Compiled Keras model
    """
    model = keras.Sequential([
        # Input layer
        layers.Input(shape=input_shape),
        
        # Normalization
        layers.BatchNormalization(),
        
        # LSTM layers
        layers.LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3),
        layers.BatchNormalization(),
        
        layers.LSTM(64, return_sequences=True, dropout=0.3, recurrent_dropout=0.3),
        layers.BatchNormalization(),
        
        layers.LSTM(32, return_sequences=False, dropout=0.3, recurrent_dropout=0.3),
        layers.BatchNormalization(),
        
        # Dense layers
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.BatchNormalization(),
        
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        
        # Output layer
        layers.Dense(num_classes, activation='softmax')
    ])
    
    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy', 'top_k_categorical_accuracy']
    )
    
    return model


In [ ]:
# Build model (uncomment after preparing dataset)
# input_shape = (X_train.shape[1], X_train.shape[2])  # (sequence_length, num_features)
# model = create_lstm_model(input_shape, NUM_CLASSES)
# model.summary()


In [ ]:
# Define callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(MODEL_DIR, 'best_model.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    keras.callbacks.CSVLogger(
        filename=os.path.join(OUTPUT_DIR, 'training_log.csv'),
        append=True
    )
]


In [ ]:
# Train the model (uncomment after building model)
# history = model.fit(
#     X_train, y_train,
#     batch_size=BATCH_SIZE,
#     epochs=EPOCHS,
#     validation_data=(X_val, y_val),
#     callbacks=callbacks,
#     verbose=1
# )


## Step 7: Evaluate the Model


In [ ]:
def plot_training_history(history):
    """
    Plot training history (loss and accuracy curves).
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot loss
    axes[0].plot(history.history['loss'], label='Training Loss')
    axes[0].plot(history.history['val_loss'], label='Validation Loss')
    axes[0].set_title('Model Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    # Plot accuracy
    axes[1].plot(history.history['accuracy'], label='Training Accuracy')
    axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy')
    axes[1].set_title('Model Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'), dpi=300, bbox_inches='tight')
    plt.show()

# Plot training history (uncomment after training)
# plot_training_history(history)


In [ ]:
def evaluate_model(model, X_test, y_test, label_encoder):
    """
    Evaluate the model on test set and print detailed metrics.
    """
    # Predictions
    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    y_true = np.argmax(y_test, axis=1)
    
    # Decode labels
    y_true_decoded = label_encoder.inverse_transform(y_true)
    y_pred_decoded = label_encoder.inverse_transform(y_pred)
    
    # Calculate accuracy
    test_loss, test_accuracy, test_top_k = model.evaluate(X_test, y_test, verbose=0)
    
    print(f"\nTest Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Test Top-K Accuracy: {test_top_k:.4f}")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_true_decoded, y_pred_decoded))
    
    # Confusion matrix
    cm = confusion_matrix(y_true_decoded, y_pred_decoded)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=sorted(label_encoder.classes_),
                yticklabels=sorted(label_encoder.classes_))
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    return test_accuracy, y_pred_decoded, y_true_decoded

# Evaluate model (uncomment after training)
# test_accuracy, y_pred, y_true = evaluate_model(model, X_test, y_test, label_encoder)


## Step 8: Save the Model


In [ ]:
# Save the final model
# model.save(os.path.join(MODEL_DIR, 'sign_language_numbers_model.h5'))
# model.save(os.path.join(MODEL_DIR, 'sign_language_numbers_model'))  # SavedModel format

# Save model summary
# with open(os.path.join(MODEL_DIR, 'model_summary.txt'), 'w') as f:
#     model.summary(print_fn=lambda x: f.write(x + '\n'))

# print(f"\nModel saved to {MODEL_DIR}")


## Step 9: Download the Model

After training, download the model files:


In [ ]:
# Create a zip file with all model files
# !cd {MODEL_DIR} && zip -r /content/sign_language_model.zip .

# Download the zip file
# from google.colab import files
# files.download('/content/sign_language_model.zip')

# Or download individual files:
# files.download(os.path.join(MODEL_DIR, 'sign_language_numbers_model.h5'))
# files.download(os.path.join(MODEL_DIR, 'label_encoder.pkl'))


## Step 10: Inference Example

Example code to use the trained model for prediction:


In [ ]:
def predict_sign_language_number(video_path, model, label_encoder):
    """
    Predict the sign language number from a video.
    
    Args:
        video_path: Path to video file
        model: Trained Keras model
        label_encoder: Label encoder used during training
    
    Returns:
        predicted_number: Predicted number
        confidence: Confidence score
    """
    # Extract landmarks
    landmarks = extract_landmarks_from_video(video_path)
    
    # Reshape for model input (add batch dimension)
    landmarks = np.expand_dims(landmarks, axis=0)
    
    # Predict
    predictions = model.predict(landmarks, verbose=0)
    
    # Get predicted class
    predicted_class_idx = np.argmax(predictions[0])
    confidence = predictions[0][predicted_class_idx]
    
    # Decode label
    predicted_number = label_encoder.inverse_transform([predicted_class_idx])[0]
    
    return predicted_number, confidence

# Example usage:
# Load model and label encoder
# model = keras.models.load_model(os.path.join(MODEL_DIR, 'sign_language_numbers_model.h5'))
# with open(os.path.join(MODEL_DIR, 'label_encoder.pkl'), 'rb') as f:
#     label_encoder = pickle.load(f)

# Predict from a video
# predicted_number, confidence = predict_sign_language_number('path/to/video.mp4', model, label_encoder)
# print(f"Predicted Number: {predicted_number}, Confidence: {confidence:.4f}")
